In [1]:
import gym
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from itertools import product

import warnings
warnings.filterwarnings("ignore")

def warn(*args, **kwargs):
    return None

warnings.warn = warn

In [2]:
env = gym.make('Taxi-v3')

In [3]:
discount = 1.0    # Discounting rate
step_size = 0.99  # Learning rate
epsilon = 1.0     # Exploration rate
epsilon_final = 0.01
epsilon_decay = 0.01
num_actions = env.action_space.n
num_states = env.observation_space.n

# Funções

In [92]:
def test_agent(env, agent, goal, max_episodes=10000):
    """
    Testa o agente no ambiente até que o total de recompensas atinja o objetivo ou o número máximo de episódios seja alcançado.

    Parâmetros:
    env: O ambiente do Gym.
    agent: O agente que está sendo testado.
    goal: O objetivo de recompensas totais que deve ser alcançado.
    max_episodes: Número máximo de episódios para evitar loops infinitos.

    Retorna:
    list_rewards: Uma lista com as recompensas acumuladas por episódio.
    """

    total_rewards = 0  # Inicializa o total de recompensas
    list_rewards = []  # Inicializa a lista de recompensas acumuladas
    episodes = 0       # Contador de episódios

    # Jogar episódios até que as recompensas totais alcancem o objetivo ou o limite de episódios seja atingido
    for episode in tqdm(range(max_episodes), desc="Treinando o agente"):
        if total_rewards >= goal:
            print(f"Objetivo alcançado em {episodes} episódios.")
            break

        obs = env.reset()                # Reiniciar o ambiente para um novo episódio
        action = agent.agent_start(obs)  # Iniciar o agente com a observação inicial
        done = False                     # Reiniciar a flag de término para cada episódio
        episode_reward = 0               # Reiniciar a recompensa do episódio

        # Loop através de cada passo no episódio
        while not done:
            obs, reward, terminated, _ = env.step(action)  # Executar um passo no ambiente
            episode_reward += reward  # Acumular recompensa do episódio

            if terminated:
                agent.agent_end(reward)  # Encerrar o episódio do agente
                done = True              # Definir a flag de término para sair do loop
            else:
                action = agent.agent_step(reward, obs)  # Continuar o passo do agente

        total_rewards += episode_reward     # Acumular recompensas
        episodes += 1                       # Contar os episódios
        list_rewards.append(total_rewards)  # Adicionar recompensas totais à lista

    # Fechar o ambiente
    env.close()

    # Imprimir resultados finais
    print(f"Episódios: {episodes}")
    print(f"Precisão: {1 - (episodes / total_rewards) :.2f}")
    print(f"Recompensas totais: {sum(list_rewards)}")

    return list_rewards

In [16]:
def plot_rewards_cumulative(rewards, window_size=100):
    """
    Plota as recompensas acumuladas ao longo dos episódios.

    Parâmetros:
    rewards: Lista de recompensas acumuladas por episódio.
    """
    cumulative_rewards = [sum(rewards[:i+1]) for i in range(len(rewards))]

    # Calcular a média móvel para suavizar a curva
    window_size = window_size
    moving_avg = np.convolve(cumulative_rewards, np.ones(window_size)/window_size, mode='valid')

    plt.figure(figsize=(12, 6))  # Tamanho da figura
    plt.plot(cumulative_rewards, label='Recompensa Acumulada', color='blue')
    plt.plot(range(window_size-1, len(cumulative_rewards)), moving_avg, label=f'Média Móvel ({window_size} episódios)', color='orange', linestyle='--')

    plt.xlabel("Episódios", fontsize=14)
    plt.ylabel("Recompensa Acumulada", fontsize=14)
    plt.title("Recompensa Acumulada ao Longo dos Episódios", fontsize=16)

    plt.grid(True)  # Adicionar grade ao gráfico
    plt.legend(fontsize=12)  # Adicionar legenda

    # Adicionar anotações para pontos específicos, como o início e o fim
    plt.annotate(f'Início', xy=(0, cumulative_rewards[0]), xytext=(50, cumulative_rewards[0] + 50),
                 arrowprops=dict(facecolor='black', shrink=0.05))
    plt.annotate(f'Fim', xy=(len(cumulative_rewards) - 1, cumulative_rewards[-1]), xytext=(len(cumulative_rewards) - 50, cumulative_rewards[-1] - 50),
                 arrowprops=dict(facecolor='black', shrink=0.05))

    plt.show()  # Mostrar o gráfico

In [23]:
def run_episode(env, agent, max_steps):
    state = env.reset()
    total_reward = 0
    for _ in range(max_steps):
        action = agent.agent_start(state) if _ == 0 else agent.agent_step(reward, state)
        state, reward, done, _ = env.step(action)
        total_reward += reward
        if done:
            agent.agent_end(reward)
            break
    return total_reward

In [22]:
# Função para executar o experimento
def run_experiment(env_name, agents, num_episodes, step_sizes, num_runs, max_steps):
    env = gym.make(env_name)
    all_reward_sums = {}
    cross_product = list(product(agents.keys(), step_sizes, range(num_runs)))

    for agent_name, step_size, run in tqdm(cross_product, desc="Executando experimentos"):
        if (agent_name, step_size) not in all_reward_sums:
            all_reward_sums[(agent_name, step_size)] = []

        agent_class = agents[agent_name]
        agent_info = {
            "env": env,
            "num_actions": env.action_space.n,
            "num_states": env.observation_space.n,
            "epsilon": 0.1,
            "discount": 1.0,
            "step_size": step_size,
            "seed": run
        }
        agent = agent_class(**agent_info)

        total_rewards = []
        for episode in range(num_episodes):
            total_reward = run_episode(env, agent, max_steps)
            total_rewards.append(total_reward)
        all_reward_sums[(agent_name, step_size)].append(np.mean(total_rewards))

    env.close()
    return all_reward_sums, step_sizes

In [24]:
def plot_results(all_reward_sums, step_sizes, algorithms):
    for algorithm in algorithms:
        algorithm_means = np.array([np.mean(all_reward_sums[(algorithm, step_size)]) for step_size in step_sizes])
        algorithm_stds = np.array([np.std(all_reward_sums[(algorithm, step_size)]) for step_size in step_sizes])
        plt.plot(step_sizes, algorithm_means, marker='o', linestyle='solid', label=algorithm)
        plt.fill_between(step_sizes, algorithm_means + algorithm_stds, algorithm_means - algorithm_stds, alpha=0.2)

    plt.legend()
    plt.xlabel("Step-size")
    plt.ylabel("Sum of\n rewards\n per episode", rotation=0, labelpad=50)
    plt.xticks(step_sizes)
    plt.show()

In [26]:
def run_episodes(env, agent, total_episodes, max_steps):
    rewards = []

    for episode in tqdm(range(total_episodes), desc="Episódios de treinamento"):
        observation = env.reset()
        done = False
        total_reward = 0
        steps = 0

        while not done and steps < max_steps:
            if steps == 0:
                action = agent.agent_start(observation)
            else:
                action = agent.agent_step(reward, observation)

            observation, reward, done, info = env.step(action)
            total_reward += reward
            steps += 1

        agent.agent_end(reward)
        rewards.append(total_reward)

    return rewards

# Agents

In [17]:
class QLearningAgent:
    def __init__(self, env, num_actions, num_states, discount, epsilon, step_size, seed=42):
        self.env = env
        self.num_actions = env.action_space.n
        self.num_states = env.observation_space.n
        self.rand_generator = np.random.RandomState(seed)
        self.discount = discount    # Fator de desconto
        self.epsilon = epsilon      # Taxa de exploração
        self.step_size = step_size  # Taxa de aprendizado
        self.seed = seed
        self.q = np.zeros((self.num_states, self.num_actions)) # Matriz de Q-values

    def agent_start(self, observation):
        state = observation
        current_q = self.q[state, :]
        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)
        self.prev_state = state
        self.prev_action = action
        return action

    def agent_step(self, reward, observation):
        state = observation
        current_q = self.q[state, :]

        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)

        target = reward + self.discount * np.max(current_q)
        self.q[self.prev_state, self.prev_action] += self.step_size * (target - self.q[self.prev_state, self.prev_action])

        self.prev_state = state
        self.prev_action = action

        return action

    def agent_end(self, reward):
        self.q[self.prev_state, self.prev_action] += self.step_size * (reward - self.q[self.prev_state, self.prev_action])

    def argmax(self, q_values):
        top = float("-inf")
        ties = []

        for i in range(len(q_values)):
            if q_values[i] > top:
                top = q_values[i]
                ties = [i]
            elif q_values[i] == top:
                ties.append(i)

        return self.rand_generator.choice(ties)

In [21]:
class ExpectedSarsaAgent:
    def __init__(self, env, num_actions, num_states, discount, epsilon, step_size, seed=42):
      # Store the parameters provided in agent_init_info.
      self.num_actions = num_actions
      self.num_states = num_states
      self.epsilon = epsilon
      self.step_size = step_size
      self.discount = discount
      self.seed = seed
      self.rand_generator = np.random.RandomState(seed)
      # Create an array for action-value estimates and initialize it to zero.
      self.q = np.zeros((self.num_states, self.num_actions)) # Matriz de Q-value

    def agent_start(self, observation):
        """The first method called when the episode starts, called after
        the environment starts.
        Args:
            observation (int): the state observation from the
                environment's evn_start function.
        Returns:
            action (int): the first action the agent takes.
        """

        # Choose action using epsilon greedy.
        state = observation
        current_q = self.q[state, :]
        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)
        self.prev_state = state
        self.prev_action = action
        return action

    def agent_step(self, reward, observation):
        """A step taken by the agent.
        Args:
            reward (float): the reward received for taking the last action taken
            observation (int): the state observation from the
                environment's step based on where the agent ended up after the
                last step.
        Returns:
            action (int): the action the agent is taking.
        """

        # Choose action using epsilon greedy.
        state = observation
        current_q = self.q[state,:]
        if self.rand_generator.rand() < self.epsilon:
            action = self.rand_generator.randint(self.num_actions)
        else:
            action = self.argmax(current_q)

        expected_q = np.sum((self.epsilon / self.num_actions) * current_q)
        best_action = self.argmax(current_q)

        expected_q += (1.0 - self.epsilon) * self.q[state, best_action]

        self.q[self.prev_state, self.prev_action] += self.step_size * (
            reward + self.discount * expected_q - self.q[self.prev_state, self.prev_action])

        self.prev_state = state
        self.prev_action = action
        return action

    def agent_end(self, reward):
        """Run when the agent terminates.
        Args:
            reward (float): the reward the agent received for entering the
                terminal state.
        """
        # Update the Q-value for the last state-action pair.
        self.q[self.prev_state, self.prev_action] += self.step_size * (reward - self.q[self.prev_state, self.prev_action])

    def argmax(self, q_values):
        """argmax with random tie-breaking
        Args:
            q_values (Numpy array): the array of action-values
        Returns:
            action (int): an action with the highest value
        """
        top = float("-inf")
        ties = []

        for i in range(len(q_values)):
            if q_values[i] > top:
                top = q_values[i]
                ties = []

            if q_values[i] == top:
                ties.append(i)

        return self.rand_generator.choice(ties)

# Tests

In [ ]:
np.random.seed(42)

# Parâmetros do experimento
env_name = "Taxi-v3"
agents = {
    "Q-learning": QLearningAgent,
    "Expected Sarsa": ExpectedSarsaAgent
}

num_episodes = 1000
step_sizes = np.linspace(0.1, 1.0, 10)
num_runs = 30
max_steps = 99

# Executar o experimento e plotar resultados
all_reward_sums, step_sizes = run_experiment(env_name, agents, num_episodes, step_sizes, num_runs, max_steps)
plot_results(all_reward_sums, step_sizes, list(agents.keys()))

# Avaliação

In [97]:
# Inicialização do agente
num_actions = env.action_space.n
num_states = env.observation_space.n
discount = 0.933
epsilon = 0.01
step_size = 0.9

agent_info = (env, num_actions, num_states, discount, epsilon, step_size)
agent = QLearningAgent(*agent_info)

**Recompensas**


-1 por etapa, a menos que outra recompensa seja acionada.

+20 entregando passageiros.

-10 executar ações de “retirada” e “retirada” ilegalmente.

Uma ação que resulte em um noop, como bater em uma parede, incorrerá na penalidade de intervalo de tempo. Noops podem ser evitados amostrando o action_maskretornado em info.

In [ ]:
# Parâmetros do teste
goal = 1000000
max_episodes = 10000

# Executar treinamento
rewards = test_agent(env, agent, goal, max_episodes)
plot_rewards_cumulative(rewards)

In [ ]:
goal = 1000000

test = test_agent(env, agent, goal, max_episodes=5000)
plot_rewards_cumulative(test)